<div style="border-left:4px solid #2980b9;padding-left:16px;margin-bottom:22px;">
  <p style="margin:0;color:#64748b;font-size:13px;letter-spacing:.7px;"><b>GREENFORGE AI · ML / DECISION SUPPORT / 03</b></p>
  <h1 style="margin:9px 0 6px;color:#1e3a8a;">☀️ When should the machine run?</h1>
  <p style="font-size:17px;color:#334155;margin:0;">Electricity-cost and grid-energy-aware scheduling using MILP</p>
</div>

**Question:** If the company begins its working day at **08:00**, could moving a flexible machine workload to a better time reduce its electricity cost and grid demand?

**At a glance:** I compare only **🌅 Shift-start baseline (08:00–10:00)**, **⚙️ forecast-informed MILP**, and **🔭 Hindsight** (the lowest historical cost with perfect information). Each completes exactly the **same two consecutive one-hour jobs**.

**Where the data is real:** Solar readings come from **SPARK IPE_PV**, and wholesale electricity prices from the **SMARD archive** from the ETL pipeline. Notebook 01 supplies its **15-minute-ahead PV prediction** for the 08:00–08:15 interval. Notebook 02 supplies **time-aware machine and solar monitoring advisories**, displayed separately from the MILP decision. Grid-related CO₂ is estimated with the **German 2024 annual electricity factor** referenced in the 03_plant_optimization EDA.

**Where assumptions remain:** SPARK does **not** provide verified movable work orders, a factory opening time, the PV electrical connection, or a customer tariff. The **08:00 shift start**, two 1.20 kW jobs and shared-PV connection are **clearly stated what-if assumptions**. The results are *not* verified savings from actual factory dispatch.

**Guide:** [📋 Scheduling assumptions](#rules) → [📥 Data](#data) → [🛠️ Monitoring](#condition) → [⚙️ MILP method](#model) → [🧪 Evaluation](#test) → [💶 Costs](#results) → [🌍 Environmental impact](#sustainability) → [🕒 Schedules](#schedule) → [💡 Conclusions](#takeaways).

<details><summary>❔ What do “MILP” and “Hindsight” mean?</summary>

**MILP** selects a feasible schedule using the information available when the decision is made. **Hindsight** solves the same problem using solar output measured after the day has ended. It is a *theoretical benchmark*, not a real scheduling policy.

</details>

<a id="rules"></a>
### ❓ 1. What rules does the scheduling experiment follow?

Imagine that a factory starts work at **08:00**, then ask whether it could run the same flexible machine work later, when electricity is less expensive or more solar energy is available.

| Assumption | What it means |
|---|---|
| 🏭 **Two consecutive jobs** | Each needs 1.20 kW for one hour; total **2.40 kWh** |
| 🌅 **Company opens at 08:00** | My *assumed* start-of-shift baseline: 08:00–10:00 |
| 🕗 **Flexible working window** | 08:00–18:00, with 15-minute start choices and a firm finish by 18:00 |
| 🔌 **Grid limit** | Assumed 4.0 kW limit, with no known additional load |
| ☀️ **Solar sharing** | The jobs may use the observed PV supply if both systems share an electrical connection (not verified) |
| 💶 **Optimization goal** | Minimize risk-weighted wholesale electricity cost plus an illustrative grid-emissions preference |
| 🌍 **Sustainability** | Compare additional grid energy and estimated CO₂ using the German 2024 annual factor |

**In simple words:** I keep the same amount of work and move **only its start time**. This is a historical scheduling experiment, not a reconstructed factory production log.

<details><summary>🛡️ Why not assume every day can be optimized perfectly?</summary>

Notebook 01 forecasts **08:00–08:15 only**. For the rest of the day (08:15–18:00), I use historical PV patterns learned **before the test period**, plus conservative low- and zero-solar scenarios. I do not treat later 15-minute predictions as if they were available at 08:00. The actual future solar output is *not used* to select the MILP schedule.

</details>

In [1]:
# ⚙️ 2. Setup — Polars for data, NumPy/SciPy for optimization, Plotly for figures.
import os
os.environ.setdefault("POLARS_MAX_THREADS", "4")
from pathlib import Path
from datetime import datetime, date, timedelta
from html import escape
import json
import math
import numpy as np
import polars as pl
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown
from scipy.optimize import milp, Bounds, LinearConstraint

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "etl_config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the GreenForge AI project folder.")

REPORT_ROOT = ROOT / "reports" / "ml_short_term" / "notebook_runs"
OUT = REPORT_ROOT / "milp_prescriptive_scheduling"
OUT.mkdir(parents=True, exist_ok=True)
# Notebook 01 (15-minute benchmark) writes directly under notebook_runs/.
# Accept the workflow subfolder too, for projects organized by output type.
FORECAST_CANDIDATES = (
    REPORT_ROOT / "pv_predictions_15min.csv",
    REPORT_ROOT / "energy_forecasting" / "pv_predictions_15min.csv",
)
FORECAST_FILE = next((path for path in FORECAST_CANDIDATES if path.is_file()),
                     FORECAST_CANDIDATES[0])
# Prefer the model selected by training-only CV rather than the test-R² winner.
# Both columns are genuine 15-minute predictions produced by Notebook 01.
FORECAST_PREDICTION_COLUMN = "forecast_cv_alternative"

# ⚠️ Portfolio assumptions, NOT verified publisher work orders or machine limits.
JOB_KW = (1.20, 1.20)
JOB_STEPS = (4, 4)                   # 4 x 15 min per job
WORK_START, WORK_END = "08:00", "18:00"
GRID_CAP_KW = 4.0
OTHER_LOAD_KW = 0.0
ASSUME_SHARED_PV = True
MIN_PV_COVERAGE = 0.80
MIN_PRIOR_SAMPLES = 8
EARLY_STARTS = ("08:00", "09:00")  # assumed company start of work
RISK_WEIGHTS = (0.15, 0.35, 0.50)  # typical, low-solar, zero-solar; research assumptions
UBA_2024_KG_CO2_PER_KWH = 0.363   # UBA 2024 factor, from 03_plant_optimization_eda.ipynb
CARBON_SHADOW_EUR_PER_KG = 0.02   # assumed green preference in the MILP, NOT an actual tax
UBA_SOURCE_URL = "https://www.umweltbundesamt.de/themen/co2-emissionen-pro-kilowattstunde-strom-2024"

ALLOWED_WEEKDAYS = (0, 1, 2, 3, 4)
BLOCKED_START_TIMES = ()
SLOT_HOURS = 0.25
N_SLOTS = 40
TOTAL_JOB_KWH = sum(k * n * SLOT_HOURS for k, n in zip(JOB_KW, JOB_STEPS))

assert abs(sum(RISK_WEIGHTS) - 1) < 1e-12
assert len(JOB_KW) == len(JOB_STEPS) == 2
assert OTHER_LOAD_KW + max(JOB_KW) <= GRID_CAP_KW

BLUE, ORANGE, GREEN, SLATE = "#2563eb", "#f59e0b", "#16a34a", "#64748b"
PURPLE, TEAL = "#7c3aed", "#0891b2"


def wall_clock(frame: pl.DataFrame, name: str) -> pl.DataFrame:
    """Preserve the original local publisher times; handle text and zoned timestamps."""
    dtype = frame.schema[name]
    if dtype == pl.String:
        expression = pl.col(name).str.to_datetime(strict=False)
    elif isinstance(dtype, pl.Datetime) and dtype.time_zone:
        expression = (pl.col(name).dt.convert_time_zone("Europe/Berlin")
                      .dt.replace_time_zone(None))
    else:
        expression = pl.col(name).cast(pl.Datetime("us"), strict=False)
    return frame.with_columns(expression.alias(name))


def _pretty(value):
    if value is None:
        return "—"
    if isinstance(value, (float, np.floating)):
        return f"{value:.5f}" if math.isfinite(float(value)) else "—"
    if isinstance(value, (int, np.integer)):
        return f"{value:,}"
    if isinstance(value, datetime):
        return value.strftime("%d %b %Y %H:%M")
    if isinstance(value, date):
        return value.strftime("%d %b %Y")
    return str(value)


def show_table(frame: pl.DataFrame, heading=None, max_rows=35):
    """Compact dark notebook table; Polars remains the only dataframe engine."""
    if not isinstance(frame, pl.DataFrame):
        raise TypeError("show_table expects a Polars DataFrame")
    if heading:
        display(Markdown(f"**{heading}**"))
    labels = frame.columns
    header = "".join(
        f'<th style="background:#eff6ff;color:#1e3a8a;padding:8px 11px;'
        f'text-align:left !important;direction:ltr;font-weight:600;white-space:nowrap;">{escape(c)}</th>'
        for c in labels
    )
    rows = []
    for index, row in enumerate(frame.head(max_rows).iter_rows(named=True)):
        shade = "#161616" if index % 2 == 0 else "#222222"
        cells = "".join(
            f'<td style="background:{shade};color:#d1d5db;padding:7px 11px;'
            f'border-bottom:1px solid #737373;text-align:left !important;direction:ltr;vertical-align:middle;white-space:nowrap;">'
            f'{escape(_pretty(row[col]))}</td>' for col in labels
        )
        rows.append("<tr>" + cells + "</tr>")
    if not rows:
        rows.append(f'<tr><td colspan="{max(1, len(labels))}" '
                    'style="background:#161616;color:#d1d5db;padding:10px;">'
                    'No rows available.</td></tr>')
    display(HTML(
        '<div style="overflow-x:auto;max-width:100%;margin:8px 0 16px;">'
        '<table style="border-collapse:collapse;width:auto;min-width:70%;'
        'font-family:Arial,sans-serif;font-size:13px;text-align:left !important;">'
        '<thead><tr>' + header + '</tr></thead><tbody>' +
        ''.join(rows) + '</tbody></table></div>'
    ))
    if frame.height > max_rows:
        display(Markdown(f"*Showing {max_rows:,} of {frame.height:,} rows.*"))

def show_cards(items):
    cards = "".join(
        '<div style="background:#f8fafc;border:1px solid #e2e8f0;'
        'border-radius:15px;padding:19px 23px;min-width:0;box-shadow:0 2px 7px rgba(15,23,42,.04);">'
        f'<div style="color:#64748b;font-size:13px;margin-bottom:9px;">{escape(str(title))}</div>'
        f'<div style="color:#334155;font-weight:750;font-size:27px;margin-bottom:7px;">{escape(str(value))}</div>'
        f'<div style="color:#475569;font-size:12px;line-height:1.45;">{escape(str(detail))}</div>'
        '</div>' for title, value, detail in items
    )
    display(HTML('<div style="display:grid;grid-template-columns:repeat(3,minmax(190px,1fr));'
                 'gap:14px;margin:14px 0 22px;">' + cards + '</div>'))


def report(message):
    """Original theme-friendly report block, readable in dark Jupyter."""
    text = str(message).strip()
    display(Markdown(
        "> **💡 What this tells us**\n>\n" +
        "\n".join("> " + line for line in text.splitlines())
    ))

show_cards([
    ("🕒 Work window", "08:00–18:00", "15-minute possible start times"),
    ("🏭 Same workload", f"{TOTAL_JOB_KWH:.2f} kWh", "Two continuous one-hour jobs; illustrative"),
    ("☀️ PV quality", "≥80%", "Required observation coverage"),
    ("📈 Forecast", "15 min", "08:00–08:15 only; earlier PV patterns for later hours"),
    ("🎯 Target", "Lower cost", "Incremental wholesale cost, not a factory bill"),
    ("🌍 Sustainability", "UBA 2024", "0.363 kg CO₂/kWh of modeled grid imports"),
])


<a id="data"></a>
### ❓ 2. Where do the electricity prices and solar readings come from?

I read **IPE_PV solar measurements**, **historical SMARD electricity prices**, and Notebook 01's **normal 15-minute PV forecast export**. We require at least **80% measurement coverage** before treating a solar interval as usable.

**In simple words:** If a reading is missing, I leave it missing. I do not pretend the solar panels generated zero electricity.

After 08:15, the model uses solar patterns learned from **earlier dates**. It does not have cloud forecasts for the rest of the workday. The actual PV readings are used later only to judge how the plan performed.

<details><summary>☁️ Why are historical solar patterns needed?</summary>

Notebook 01 predicts only the upcoming **08:00–08:15** interval. To compare possible starts through 18:00, the MILP still needs an estimate for the other 39 slots. Historical patterns are a simple assumption, **not a replacement for a proper day-ahead weather forecast**.

</details>

In [12]:
# 📥 3. Load and validate with Polars. Missing solar readings remain missing.
if not FORECAST_FILE.is_file():
    raise FileNotFoundError(
        f"Missing 15-minute forecast: {FORECAST_FILE}. "
        "Run Notebook 01's normal forecast export; no one-hour TEST is needed."
    )

PRICE_COL = "DayAhead_Price_EUR_MWh"


def read_features(system, extra=()):
    path = ROOT / "data" / "processed" / "features" / system / "eda_15min.parquet"
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; run the ETL pipeline first.")
    scan = pl.scan_parquet(path)
    available = set(scan.collect_schema().names())
    mandatory = {"WsDateTime"}
    if system == "IPE_PV":
        mandatory.update(("power_kw", "coverage"))
    if missing := mandatory - available:
        raise ValueError(f"{path.name}: missing {sorted(missing)}")
    cols = sorted(mandatory | (set(extra) & available))
    frame = wall_clock(scan.select(cols).collect(), "WsDateTime")
    if (frame.get_column("WsDateTime").null_count()
            or frame.get_column("WsDateTime").is_duplicated().any()):
        raise ValueError(f"{system}: missing or repeated timestamps")
    return frame


pv = read_features("IPE_PV", (PRICE_COL,)).rename({"power_kw": "pv_measured_kw"})
tec = read_features("TEC_48S", (PRICE_COL,))
if PRICE_COL in pv.columns:
    pv = pv.rename({PRICE_COL: "pv_price"})
if PRICE_COL in tec.columns:
    tec = tec.rename({PRICE_COL: "tec_price"})
tec = tec.select([c for c in ("WsDateTime", "tec_price") if c in tec.columns])
archive = pv.join(tec, on="WsDateTime", how="full", coalesce=True)
for col in ("pv_measured_kw", "coverage", "pv_price", "tec_price"):
    if col not in archive.columns:
        archive = archive.with_columns(pl.lit(None, dtype=pl.Float64).alias(col))
    archive = archive.with_columns(pl.col(col).cast(pl.Float64, strict=False).fill_nan(None))
conflict = (pl.col("pv_price").is_not_null() & pl.col("tec_price").is_not_null()
            & ((pl.col("pv_price") - pl.col("tec_price")).abs() > 0.0001))
archive = (archive.with_columns(
    pl.when(conflict).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.coalesce("pv_price", "tec_price")).alias("price_eur_mwh"),
    pl.when(pl.col("pv_measured_kw") < 0).then(pl.lit(None, dtype=pl.Float64))
      .otherwise(pl.col("pv_measured_kw")).alias("pv_measured_kw"),
).with_columns(
    pl.when(conflict).then(pl.lit("conflicting archive prices"))
      .when(pl.col("price_eur_mwh").is_not_null()).then(pl.lit("ETL SMARD archive"))
      .otherwise(pl.lit("unavailable")).alias("price_source")
).sort("WsDateTime"))

forecast_raw = pl.read_csv(FORECAST_FILE)
required = {"WsDateTime", "target_time", "target", "prediction"}
if missing := required - set(forecast_raw.columns):
    raise ValueError(f"Notebook 01 15-minute CSV is missing {sorted(missing)}")

selected_prediction = (FORECAST_PREDICTION_COLUMN
                       if FORECAST_PREDICTION_COLUMN in forecast_raw.columns
                       else "prediction")
for column in ("WsDateTime", "target_time"):
    forecast_raw = wall_clock(forecast_raw, column)
forecast_raw = forecast_raw.with_columns([
    pl.col("target").cast(pl.Float64, strict=False).fill_nan(None),
    pl.col(selected_prediction).cast(pl.Float64, strict=False)
      .fill_nan(None).alias("pv_forecast_kw"),
])
if forecast_raw.select([pl.col("WsDateTime").is_null().any(),
                        pl.col("target_time").is_null().any()]).row(0) != (False, False):
    raise ValueError("Notebook 01 predictions contain invalid timestamps")
if forecast_raw.select(pl.col("target_time").is_duplicated().any()).item():
    raise ValueError("Notebook 01 contains duplicate target timestamps")

# In Notebook 01, WsDateTime identifies the last completed 15-minute
# measurement interval. Its forecast becomes available at WsDateTime + 15 min,
# and predicts the next interval beginning exactly at target_time.
forecast_raw = forecast_raw.with_columns(
    (pl.col("WsDateTime") + pl.duration(minutes=15)).alias("issue_time")
)
if forecast_raw.filter(pl.col("target_time") != pl.col("issue_time")).height:
    raise ValueError("Notebook 01 CSV is not aligned to a 15-minute next-interval forecast")

# Only the one value that was available at 08:00 is fed to the MILP.
# Later workday PV is estimated solely from pre-test PV history.
forecast = (forecast_raw.filter(pl.col("issue_time").dt.strftime("%H:%M") == WORK_START)
            .filter(pl.col("target").is_not_null()
                    & pl.col("target").is_finite()
                    & pl.col("pv_forecast_kw").is_not_null()
                    & pl.col("pv_forecast_kw").is_finite())
            .select(["issue_time", "target_time", "target", "pv_forecast_kw"])
            .sort("issue_time"))
if forecast.is_empty():
    raise ValueError("No 08:00 15-minute PV forecasts found. Run Notebook 01 first.")

# Solar profiles learned from STRICTLY PRE-TEST observations.
first_test_issue = forecast.get_column("issue_time").min()
reference = (archive.filter(
    (pl.col("WsDateTime") < first_test_issue)
    & (pl.col("coverage") >= MIN_PV_COVERAGE)
    & pl.col("pv_measured_kw").is_not_null()
    & pl.col("pv_measured_kw").is_finite()
).with_columns(pl.col("WsDateTime").dt.strftime("%H:%M").alias("clock")))
if reference.is_empty():
    raise ValueError("No earlier valid PV measurements available for historical profiles")
prior = reference.group_by("clock").agg(
    pl.len().alias("n"),
    pl.col("pv_measured_kw").clip(lower_bound=0).median().alias("median"),
    pl.col("pv_measured_kw").clip(lower_bound=0).quantile(.25).alias("low"),
)
prior_lookup = {
    row["clock"]: (float(row["median"]), float(row["low"]))
    if row["n"] >= MIN_PRIOR_SAMPLES else (0.0, 0.0)
    for row in prior.iter_rows(named=True)
}
archive_lookup = {row["WsDateTime"]: row for row in archive.iter_rows(named=True)}

show_table(pl.DataFrame([
    {"Input": "☀️ PV readings", "Rows": pv.height, "What it is": "IPE_PV cleaned ETL"},
    {"Input": "💶 Price readings", "Rows": archive.get_column("price_eur_mwh").is_not_null().sum(),
     "What it is": "Archived SMARD from ETL"},
    {"Input": "🕗 08:00 forecast dates", "Rows": forecast.get_column("issue_time").n_unique(),
     "What it is": f"Notebook 01 next-15-minute forecast ({selected_prediction})"},
    {"Input": "📚 Older PV readings", "Rows": reference.height,
     "What it is": "Known before test period"},
]), "Data sources")
report("Notebook 01 provides a **real 15-minute-ahead forecast** for 08:00–08:15. "
       "The remaining 39 workday intervals use PV patterns learned **before** the test period. "
       "Those patterns are not 15-minute forecasts. I keep missing readings missing and score decisions only after the day ends.")


**Data sources**

Input,Rows,What it is
☀️ PV readings,"35,136",IPE_PV cleaned ETL
💶 Price readings,"35,128",Archived SMARD from ETL
🕗 08:00 forecast dates,17,Notebook 01 next-15-minute forecast (forecast_cv_alternative)
📚 Older PV readings,"6,656",Known before test period


> **💡 What this tells us**
>
> Notebook 01 provides a **real 15-minute-ahead forecast** for 08:00–08:15. The remaining 39 workday intervals use PV patterns learned **before** the test period. Those patterns are not 15-minute forecasts. I keep missing readings missing and score decisions only after the day ends.

<a id="condition"></a>
### 🛠️ Monitoring check — what was known before scheduling?

Notebook 02 provides **measurement-quality checks** and **observations to review** for the lathe and solar system. I read its exported `anomaly_review_15min.parquet` and look only at records whose **`available_at` is no later than 08:00** on each scheduling day.

A recent alert means **an engineer may want to investigate**; it is not proof of a fault. Missing or old measurements are shown as *unknown*, never as a clean bill of health. Because Notebook 02 reports some sustained solar-zero events only once per day, I also look back over the **previous 24 hours**.

**In simple words:** The scheduler still calculates the same costs and start times. This section adds the context a human would check *before accepting* a suggested schedule.


In [3]:

# 🛠️ Notebook 02 → Notebook 03
# Use only monitoring observations available by 08:00.
# Null review count means NOT EVALUATED, not zero alerts.

NOTEBOOK2_CANDIDATES = (
    REPORT_ROOT / "anomaly_review_15min.parquet",
    REPORT_ROOT / "anomaly_detection_degradation"
    / "anomaly_review_15min.parquet",
)

EARLY_MACHINE_CANDIDATES = (
    REPORT_ROOT / "machine_monitoring_early_assessment_15min.parquet",
    REPORT_ROOT / "anomaly_detection_degradation"
    / "machine_monitoring_early_assessment_15min.parquet",
)

NOTEBOOK2_FILE = next(
    (p for p in NOTEBOOK2_CANDIDATES if p.is_file()),
    None
)

EARLY_MACHINE_FILE = next(
    (p for p in EARLY_MACHINE_CANDIDATES if p.is_file()),
    None
)

MONITOR_LOOKBACK_HOURS = 24
MONITOR_MAX_AGE_MINUTES = 60
MONITORED_SYSTEMS = ("TEC_48S", "IPE_PV")

MONITOR_COLUMNS = [
    "system",
    "WsDateTime",
    "interval_end",
    "available_at",
    "quality_status",
    "period",
    "model_status",
    "review_flag",
    "advisory",
    "review_reason",
]


def read_monitor_file(path):
    frame = pl.read_parquet(path)

    required = set(MONITOR_COLUMNS) - {"review_reason"}

    if missing := required - set(frame.columns):
        raise ValueError(
            f"{path.name}: missing {sorted(missing)}"
        )

    if "review_reason" not in frame.columns:
        frame = frame.with_columns(
            pl.lit(None, dtype=pl.String).alias("review_reason")
        )

    frame = frame.select(MONITOR_COLUMNS)

    for name in ("WsDateTime", "interval_end", "available_at"):
        frame = wall_clock(frame, name)

    return frame.with_columns([
        pl.col("review_flag")
          .cast(pl.Boolean, strict=False)
          .fill_null(False),

        *[
            pl.col(col).cast(pl.String, strict=False)
            for col in (
                "system",
                "quality_status",
                "period",
                "model_status",
                "advisory",
                "review_reason",
            )
        ],
    ])


if NOTEBOOK2_FILE is not None:
    monitor = read_monitor_file(NOTEBOOK2_FILE)

    if EARLY_MACHINE_FILE is not None:
        early = read_monitor_file(EARLY_MACHINE_FILE)

        if early.filter(
            pl.col("system") != "TEC_48S"
        ).height:
            raise ValueError(
                "Early machine assessment contains another system."
            )

        # Replace only overlapping machine timestamps.
        # Preserve Notebook 02's original PV monitoring.
        original = monitor.join(
            early.select("system", "WsDateTime"),
            on=["system", "WsDateTime"],
            how="anti",
        )

        monitor = pl.concat(
            [original, early],
            how="vertical_relaxed"
        )

    invalid = monitor.filter(
        pl.col("WsDateTime").is_null()
        | pl.col("interval_end").is_null()
        | pl.col("available_at").is_null()
        | (pl.col("available_at") < pl.col("interval_end"))
        | (pl.col("interval_end") <= pl.col("WsDateTime"))
    )

    if invalid.height:
        raise ValueError(
            "Monitoring timestamps are invalid or available too early."
        )

    if monitor.select([
        "system", "WsDateTime"
    ]).is_duplicated().any():
        raise ValueError(
            "Duplicate monitoring observations."
        )

    monitor_by_system = {
        sid: (
            monitor.filter(pl.col("system") == sid)
            .sort(["available_at", "WsDateTime"])
        )
        for sid in MONITORED_SYSTEMS
    }

else:
    monitor_by_system = {}


def known_monitoring_at(system, issue):
    """Return only information known before the scheduling decision."""

    empty = {
        "monitoring_status": "Monitoring unavailable",
        "latest_available_at": None,
        "age_minutes": None,
        "reviews_last_24h": None,
        "latest_review_reason": "—",
    }

    history = monitor_by_system.get(system)

    if history is None or history.is_empty():
        return empty

    known = history.filter(
        (pl.col("available_at") <= issue)
        & (pl.col("interval_end") <= issue)
    )

    if known.is_empty():
        return empty

    latest = known.row(-1, named=True)

    age = (
        issue - latest["available_at"]
    ).total_seconds() / 60

    quality = str(
        latest.get("quality_status") or "unknown"
    )

    last_day = known.filter(
        pl.col("available_at")
        > issue - timedelta(hours=MONITOR_LOOKBACK_HOURS)
    )

    # Count only observations genuinely assessed by the model.
    assessed = last_day.filter(
        pl.col("period") == "assessment",
        pl.col("model_status") == "scored",
        pl.col("quality_status") == "valid",
    )

    reviewed = assessed.filter(
        pl.col("review_flag")
    )

    # None is different from a real zero.
    review_count = (
        int(reviewed.height)
        if not assessed.is_empty()
        else None
    )

    if age > MONITOR_MAX_AGE_MINUTES:
        status = "Monitoring stale"

    elif quality != "valid":
        status = "Measurement unavailable"

    elif review_count is None:
        status = "Not assessed by model"

    elif review_count > 0:
        status = "Review recommended (last 24h)"

    else:
        status = "No recent review flag"

    reason = (
        str(
            reviewed.row(-1, named=True).get(
                "review_reason"
            ) or "Review observation"
        )[:120]
        if not reviewed.is_empty()
        else "—"
    )

    return {
        "monitoring_status": status,
        "latest_available_at": latest["available_at"],
        "age_minutes": round(age, 1),
        "reviews_last_24h": review_count,
        "latest_review_reason": reason,
    }


monitoring_rows = []

for issue in (
    forecast.get_column("issue_time")
    .unique()
    .sort()
    .to_list()
):

    if issue.weekday() not in ALLOWED_WEEKDAYS:
        continue

    for sid in MONITORED_SYSTEMS:
        monitoring_rows.append({
            "date": issue.date(),
            "decision_time": issue,
            "system": sid,
            **known_monitoring_at(sid, issue),
        })

monitoring_context = pl.DataFrame(monitoring_rows)

# Format only the displayed table.
# Keep the underlying count numeric and nullable.
monitoring_display = monitoring_context.with_columns(
    pl.when(pl.col("reviews_last_24h").is_null())
      .then(pl.lit("Not evaluated"))
      .otherwise(
          pl.col("reviews_last_24h").cast(pl.String)
      )
      .alias("reviews_last_24h")
)

show_cards([
    (
        "🛠️ Notebook 02",
        "Connected" if NOTEBOOK2_FILE else "Not found",
        "Historical monitoring; flags are advisory",
    ),
    (
        "🕗 Decision time",
        "08:00",
        "Completed measurements only",
    ),
    (
        "🏭 Machine assessment",
        "Available" if EARLY_MACHINE_FILE else "Not added",
        "Separate earlier-trained model, if available",
    ),
])

show_table(
    monitoring_display.select([
        "date",
        "system",
        "monitoring_status",
        "latest_available_at",
        "age_minutes",
        "reviews_last_24h",
    ]),
    "🛠️ Monitoring known at the 08:00 decision",
    max_rows=12,
)

report(
    "A **0** means the model assessed observations "
    "during the previous 24 hours and recorded no review flags. "
    "**Not evaluated** means there were no model-scored "
    "assessment observations in that period. "
    "Neither result confirms equipment health or safety."
)


**🛠️ Monitoring known at the 08:00 decision**

date,system,monitoring_status,latest_available_at,age_minutes,reviews_last_24h
25 Mar 2024,TEC_48S,No recent review flag,25 Mar 2024 08:00,0.00000,0
25 Mar 2024,IPE_PV,Review recommended (last 24h),25 Mar 2024 08:00,0.00000,3
26 Mar 2024,TEC_48S,No recent review flag,26 Mar 2024 08:00,0.00000,0
26 Mar 2024,IPE_PV,Review recommended (last 24h),26 Mar 2024 08:00,0.00000,13
27 Mar 2024,TEC_48S,No recent review flag,27 Mar 2024 08:00,0.00000,0
27 Mar 2024,IPE_PV,Review recommended (last 24h),27 Mar 2024 08:00,0.00000,2
28 Mar 2024,TEC_48S,No recent review flag,28 Mar 2024 08:00,0.00000,0
28 Mar 2024,IPE_PV,No recent review flag,28 Mar 2024 08:00,0.00000,0
29 Mar 2024,TEC_48S,No recent review flag,29 Mar 2024 08:00,0.00000,0
29 Mar 2024,IPE_PV,Review recommended (last 24h),29 Mar 2024 08:00,0.00000,11


*Showing 12 of 22 rows.*

> **💡 What this tells us**
>
> A **0** means the model assessed observations during the previous 24 hours and recorded no review flags. **Not evaluated** means there were no model-scored assessment observations in that period. Neither result confirms equipment health or safety.

<a id="model"></a>
### ⚙️ 3. How does MILP choose when the machine should run?

**MILP** (Mixed-Integer Linear Programming) selects a feasible start for each job using an estimated electricity-cost objective with an assumed preference for lower grid emissions. The model uses binary choices, an objective and scheduling constraints.

The calculation below runs the model first. **After it finishes, a short table explains its three main parts** in everyday language, using our actual scheduling settings.

<details><summary>🔭 Why is Hindsight sometimes cheaper?</summary>

Hindsight uses solar measurements from **after** the scheduling decision. The main MILP does not know those future measurements when choosing a start time. Both follow the same work-window and job constraints, so Hindsight is a theoretical reference, not a strategy a factory could use in advance.

</details>


In [4]:
# ⚙️ 4. Same cost accounting and MILP constraints for every strategy.
# Candidate tables are Polars; NumPy arrays are used only inside SciPy's solver.

def candidate_jobs(times, prices, pv_kw, power_kw, duration):
    """Cost/feasibility for one job; PV supplied by caller (forecast or measured)."""
    n = len(times)
    prices = np.asarray(prices, dtype=float)
    pv_kw = np.maximum(np.asarray(pv_kw, dtype=float), 0.0)
    if n != N_SLOTS or len(prices) != n or len(pv_kw) != n:
        raise ValueError("Scheduling needs the complete 08:00–18:00 grid")
    if not (np.isfinite(prices).all() and np.isfinite(pv_kw).all()):
        raise ValueError("Nonfinite price or solar estimate")
    baseline = np.maximum(OTHER_LOAD_KW - (pv_kw if ASSUME_SHARED_PV else 0), 0)
    rows = []
    for start in range(n - duration + 1):
        clock = times[start].strftime("%H:%M")
        demand = OTHER_LOAD_KW + power_kw
        after = np.maximum(demand - (pv_kw[start:start+duration] if ASSUME_SHARED_PV else 0), 0)
        extra_grid_kwh = np.maximum(after - baseline[start:start+duration], 0) * SLOT_HOURS
        rows.append({
            "index": start, "start": times[start],
            "end": times[start+duration-1] + timedelta(minutes=15),
            "clock": clock,
            "feasible": demand <= GRID_CAP_KW + 1e-9 and clock not in BLOCKED_START_TIMES,
            "cost_eur": float(np.dot(extra_grid_kwh, prices[start:start+duration]) / 1000),
            "grid_kwh": float(extra_grid_kwh.sum()),
        })
    return pl.DataFrame(rows)


def solve_two_jobs(first: pl.DataFrame, second: pl.DataFrame,
                   column="cost_eur", prefer="earliest"):
    """Exactly two back-to-back binary decisions with independent optimality check."""
    n1, n2 = first.height, second.height
    s1 = first.get_column("index").to_numpy()
    s2 = second.get_column("index").to_numpy()
    costs = np.r_[first.get_column(column).to_numpy(), second.get_column(column).to_numpy()]
    v1 = first.get_column("feasible").to_numpy()
    v2 = second.get_column("feasible").to_numpy()
    valid_pairs = [(i, j) for i in np.flatnonzero(v1) for j in np.flatnonzero(v2)
                   if s2[j] == s1[i] + JOB_STEPS[0]]
    if not valid_pairs:
        return None
    n = n1 + n2
    choose_one = np.zeros((2, n))
    choose_one[0, :n1] = 1
    choose_one[1, n1:] = 1
    occupancy = np.zeros((N_SLOTS, n))
    for i, start in enumerate(s1):
        occupancy[start:start+JOB_STEPS[0], i] = 1
    for j, start in enumerate(s2):
        occupancy[start:start+JOB_STEPS[1], n1+j] = 1
    continuity = np.r_[-s1, s2].reshape(1, -1)
    constraints = [
        LinearConstraint(choose_one, [1, 1], [1, 1]),
        LinearConstraint(occupancy, -np.inf, 1),
        LinearConstraint(continuity, JOB_STEPS[0], JOB_STEPS[0]),
    ]
    bounds = Bounds(np.zeros(n), np.r_[v1, v2].astype(float))
    integrality = np.ones(n, dtype=int)
    scaled = np.asarray(costs, dtype=float) * 1_000_000
    opts = {"time_limit": 20, "mip_rel_gap": 1e-9}
    primary = milp(c=scaled, integrality=integrality, bounds=bounds,
                   constraints=constraints, options=opts)
    if not primary.success or primary.x is None:
        raise RuntimeError(f"Two-job solver failed: {primary.message}")
    secondary_cost = np.r_[s1, s2].astype(float)  # deterministic earliest-cost tie-break
    second_pass = milp(
        c=secondary_cost, integrality=integrality, bounds=bounds,
        constraints=constraints + [LinearConstraint(scaled.reshape(1, -1),
                                                   -np.inf, float(primary.fun) + .1)],
        options=opts,
    )
    solution = second_pass if second_pass.success and second_pass.x is not None else primary
    i, j = int(np.argmax(solution.x[:n1])), int(np.argmax(solution.x[n1:]))
    if (i, j) not in valid_pairs:
        raise ArithmeticError("Solver selected a pair that violates the constraints")
    actual_cost = float(costs[i] + costs[n1+j])
    possible_min = min(float(costs[a] + costs[n1+b]) for a,b in valid_pairs)
    if actual_cost > possible_min + 1e-7:
        raise ArithmeticError("MILP did not return the cheapest feasible pair")
    return (first.row(i, named=True), second.row(j, named=True)), actual_cost


def paired_cost(options, starts, column="cost_eur"):
    """Score any strategy with exactly the same job constraints."""
    selected = []
    for frame, clock in zip(options, starts):
        match = frame.filter(pl.col("feasible") & (pl.col("clock") == clock))
        if match.height != 1:
            return None
        selected.append(match.row(0, named=True))
    if selected[1]["start"] != selected[0]["end"]:
        return None
    return float(sum(row[column] for row in selected))


<a id="test"></a>
### 🧪 4. Can we compare the same job under fair conditions?

I did backtest the proposed schedule over days where **both solar measurements and archived SMARD prices** are available. The early baseline represents a company beginning its assumed two-hour work block at **08:00**. The MILP can choose a later feasible start.

- 🌅 **Early 08:00:** Always runs the two jobs from 08:00–10:00.
- ⚙️ **Forecast MILP:** Uses Notebook 01's single 08:00–08:15 prediction, plus PV patterns from before the test period for later hours.
- 🔭 **Hindsight:** Uses real solar measured *afterward* to show the theoretical minimum cost.

**In simple words:** I move the **same 2.40 kWh of work** and measure its modeled cost, extra grid electricity, and estimated CO₂. No future actual solar reading is used when MILP makes its decision.

<details><summary>⚠️ What about the unusual April solar zero readings?</summary>

April 4–8 has a sustained zero-PV episode in the historical records. I keep these measured readings when the quality checks pass, but report that period **separately**. It is not confirmed equipment failure. Missing PV readings after the publisher's last observation are excluded.

</details>

In [13]:
# 🧪 5. Historical comparison: company starts at 08:00 vs main MILP vs Hindsight.
# Real sources: SPARK PV + archived SMARD wholesale prices; job schedule remains illustrative.
records = []
skipped = {"incomplete forecast": 0, "unavailable day": 0, "invalid observations": 0}

for group in forecast.partition_by("issue_time", maintain_order=True):
    issue = group.get_column("issue_time")[0]
    if issue.weekday() not in ALLOWED_WEEKDAYS:
        continue
    # One forecast per 08:00 decision, for the 08:00–08:15 interval only.
    if group.height != 1:
        skipped["incomplete forecast"] += 1
        continue
    times = [issue + timedelta(minutes=15*i) for i in range(N_SLOTS)]
    if any(t not in archive_lookup for t in times):
        skipped["unavailable day"] += 1
        continue
    day = pl.DataFrame([archive_lookup[t] for t in times])
    coverage_ok = day.get_column("coverage").is_not_null().all() and (
        day.get_column("coverage") >= MIN_PV_COVERAGE).all()
    price_ok = (day.get_column("price_source") == "ETL SMARD archive").all()
    solar_and_price = day.select(["pv_measured_kw", "price_eur_mwh"]).to_numpy()
    aligned = group.get_column("target_time").to_list() == times[:1]
    same_readings = np.allclose(
        solar_and_price[:1, 0].astype(float),
        group.get_column("target").to_numpy().astype(float),
        atol=1e-3, rtol=0,
    ) if aligned and np.isfinite(solar_and_price[:1, 0].astype(float)).all() else False
    if (not coverage_ok or not price_ok or not np.isfinite(solar_and_price).all()
            or not aligned or not same_readings):
        skipped["invalid observations"] += 1
        continue

    clocks = [t.strftime("%H:%M") for t in times]
    typical = np.array([prior_lookup.get(c, (0.0, 0.0))[0] for c in clocks])
    low = np.array([prior_lookup.get(c, (0.0, 0.0))[1] for c in clocks])
    informed = typical.copy()
    informed[0] = max(float(group.get_column("pv_forecast_kw")[0]), 0.0)
    # 08:15–18:00 remains the pre-test PV pattern; never fill it with later
    # 15-minute predictions made after the 08:00 scheduling decision.
    measured = np.maximum(solar_and_price[:, 0].astype(float), 0)
    prices = solar_and_price[:, 1].astype(float)

    def options_for(solar):
        return [candidate_jobs(times, prices, solar, power, duration)
                for power, duration in zip(JOB_KW, JOB_STEPS)]

    # This is a forecast-time choice: actual measured future solar is NOT used here.
    forecast_options = options_for(informed)
    low_options = options_for(low)
    zero_options = options_for(np.zeros(N_SLOTS))
    scored_options = []
    for j in range(2):
        risk_cost = (
            RISK_WEIGHTS[0] * forecast_options[j].get_column("cost_eur").to_numpy()
            + RISK_WEIGHTS[1] * low_options[j].get_column("cost_eur").to_numpy()
            + RISK_WEIGHTS[2] * zero_options[j].get_column("cost_eur").to_numpy()
        )
        risk_grid = (
            RISK_WEIGHTS[0] * forecast_options[j].get_column("grid_kwh").to_numpy()
            + RISK_WEIGHTS[1] * low_options[j].get_column("grid_kwh").to_numpy()
            + RISK_WEIGHTS[2] * zero_options[j].get_column("grid_kwh").to_numpy()
        )
        combined = risk_cost + CARBON_SHADOW_EUR_PER_KG * UBA_2024_KG_CO2_PER_KWH * risk_grid
        scored_options.append(forecast_options[j].with_columns([
            pl.Series("risk_cost_eur", risk_cost),
            pl.Series("risk_grid_kwh", risk_grid),
            pl.Series("objective_eur", combined),
        ]))

    chosen = solve_two_jobs(*scored_options, column="objective_eur", prefer="earliest")
    if chosen is None:
        skipped["invalid observations"] += 1
        continue
    milp_starts = tuple(row["clock"] for row in chosen[0])

    # This section measures what each already-chosen schedule WOULD have cost.
    # Actual future measurements are NEVER used to choose the MILP start times.
    realized = options_for(measured)
    early_cost = paired_cost(realized, EARLY_STARTS, "cost_eur")
    milp_cost = paired_cost(realized, milp_starts, "cost_eur")
    early_grid = paired_cost(realized, EARLY_STARTS, "grid_kwh")
    milp_grid = paired_cost(realized, milp_starts, "grid_kwh")
    if any(v is None for v in (early_cost, milp_cost, early_grid, milp_grid)):
        raise ValueError("The early baseline or MILP schedule is infeasible under matching constraints")
    hindsight = solve_two_jobs(*realized, column="cost_eur", prefer="earliest")
    if hindsight is None:
        raise ValueError("Hindsight could not find a feasible job schedule")
    hindsight_starts = tuple(row["clock"] for row in hindsight[0])
    hindsight_cost = float(hindsight[1])
    hindsight_grid = paired_cost(realized, hindsight_starts, "grid_kwh")
    if hindsight_grid is None or hindsight_cost > min(early_cost, milp_cost) + 1e-7:
        raise ArithmeticError("Hindsight must be the lowest realized wholesale-cost solution")
    if milp_cost < hindsight_cost - 1e-7:
        raise ArithmeticError("Forecast MILP cannot beat hindsight's feasible historical minimum")

    records.append({
        "date": issue.date(),
        "period": ("Apr 4–8 extended zero-PV readings"
                   if date(2024, 4, 4) <= issue.date() < date(2024, 4, 9)
                   else "Outside extended zero-PV period"),
        "Early_starts": " + ".join(EARLY_STARTS),
        "MILP_starts": " + ".join(milp_starts),
        "Hindsight_starts": " + ".join(hindsight_starts),
        "Early_cost_EUR": early_cost,
        "MILP_cost_EUR": milp_cost,
        "Hindsight_cost_EUR": hindsight_cost,
        "Early_grid_kWh": early_grid,
        "MILP_grid_kWh": milp_grid,
        "Hindsight_grid_kWh": hindsight_grid,
        "MILP_estimated_objective_EUR": float(chosen[1]),
    })

if not records:
    raise ValueError("No matching complete historical days; verify forecast and ETL data. "
                     f"Skipped: {skipped}. Missing intervals were never filled.")

results = pl.DataFrame(records).with_columns([
    (pl.col("Early_cost_EUR") - pl.col("MILP_cost_EUR")).alias("MILP_saving_vs_early_EUR"),
    (pl.col("MILP_cost_EUR") - pl.col("Hindsight_cost_EUR")).alias("MILP_regret_EUR"),
    (pl.col("Early_grid_kWh") - pl.col("MILP_grid_kWh")).alias("Grid_reduction_vs_early_kWh"),
    *[(pl.col(f"{s}_grid_kWh") * UBA_2024_KG_CO2_PER_KWH).alias(f"{s}_CO2_kg")
      for s in ("Early", "MILP", "Hindsight")],
]).with_columns(
    (pl.col("Early_CO2_kg") - pl.col("MILP_CO2_kg")).alias("CO2_reduction_vs_early_kg")
).sort("date")

report(f"I evaluated **{results.height} matching historical days** using the **08:00–08:15** "
       f"forecast exported by Notebook 01 ({selected_prediction}). All strategies perform the same "
       "illustrative work and are scored against identical recorded solar and wholesale-price data. "
       "The remaining workday solar input is a pre-test historical estimate, not a full-day forecast. "
       "Future measured solar is used only for evaluation and Hindsight.")

> **💡 What this tells us**
>
> I evaluated **11 matching historical days** using the **08:00–08:15** forecast exported by Notebook 01 (forecast_cv_alternative). All strategies perform the same illustrative work and are scored against identical recorded solar and wholesale-price data. The remaining workday solar input is a pre-test historical estimate, not a full-day forecast. Future measured solar is used only for evaluation and Hindsight.

In [6]:

# 🛠️ Monitoring context for the evaluated scheduling days.
# Display unavailable assessment counts as "Not evaluated".

tec_context = (
    monitoring_context
    .filter(pl.col("system") == "TEC_48S")
    .select([
        "date",
        pl.col("monitoring_status").alias(
            "Machine_monitoring"
        ),
        pl.col("reviews_last_24h").alias(
            "Machine_reviews_24h"
        ),
    ])
)

pv_context = (
    monitoring_context
    .filter(pl.col("system") == "IPE_PV")
    .select([
        "date",
        pl.col("monitoring_status").alias(
            "PV_monitoring"
        ),
        pl.col("reviews_last_24h").alias(
            "PV_reviews_24h"
        ),
    ])
)

results = (
    results
    .join(tec_context, on="date", how="left")
    .join(pv_context, on="date", how="left")
)

# Presentation only: preserve nullable numeric counts in results.
summary_view = results.select([
    "date",
    "Machine_monitoring",
    "PV_monitoring",
    "Machine_reviews_24h",
    "PV_reviews_24h",
]).with_columns([
    pl.when(pl.col(name).is_null())
      .then(pl.lit("Not evaluated"))
      .otherwise(pl.col(name).cast(pl.String))
      .alias(name)
    for name in (
        "Machine_reviews_24h",
        "PV_reviews_24h",
    )
])

show_table(
    summary_view,
    "🛠️ Monitoring context for the evaluated scheduling days",
    max_rows=18,
)

review_days = results.filter(
    (
        pl.col("Machine_reviews_24h").fill_null(0) > 0
    )
    | (
        pl.col("PV_reviews_24h").fill_null(0) > 0
    )
).height

machine_not_evaluated = results.filter(
    pl.col("Machine_reviews_24h").is_null()
).height

report(
    f"Across **{results.height} scheduling dates**, "
    f"**{review_days}** had observations flagged for review. "
    f"Machine assessment was unavailable on "
    f"**{machine_not_evaluated} dates**. "
    "A zero means no flagged readings in an assessed window, "
    "not proof of good equipment condition. "
    "Monitoring information supports human review "
    "and does not alter the MILP decision."
)


**🛠️ Monitoring context for the evaluated scheduling days**

date,Machine_monitoring,PV_monitoring,Machine_reviews_24h,PV_reviews_24h
25 Mar 2024,No recent review flag,Review recommended (last 24h),0,3
26 Mar 2024,No recent review flag,Review recommended (last 24h),0,13
27 Mar 2024,No recent review flag,Review recommended (last 24h),0,2
28 Mar 2024,No recent review flag,No recent review flag,0,0
29 Mar 2024,No recent review flag,Review recommended (last 24h),0,11
01 Apr 2024,No recent review flag,Review recommended (last 24h),0,4
02 Apr 2024,No recent review flag,No recent review flag,0,0
03 Apr 2024,No recent review flag,Review recommended (last 24h),0,4
04 Apr 2024,No recent review flag,No recent review flag,0,0
05 Apr 2024,No recent review flag,Review recommended (last 24h),0,1


> **💡 What this tells us**
>
> Across **11 scheduling dates**, **8** had observations flagged for review. Machine assessment was unavailable on **0 dates**. A zero means no flagged readings in an assessed window, not proof of good equipment condition. Monitoring information supports human review and does not alter the MILP decision.

In [ ]:
# ⚙️ The three parts of our MILP — shown after the scheduling model runs.
# This explanation uses the exact settings above; it does not change the solution.

milp_explanation = pl.DataFrame([
    {
        "MILP part": "1️⃣ Decision variables",
        "What it means": "What start times can we choose?",
        "How it works here": "A yes/no decision for each 15-minute start time of each job.",
    },
    {
        "MILP part": "2️⃣ Objective function",
        "What it means": "What do we want to minimize?",
        "How it works here": (
            "Estimated electricity cost + a small CO₂ preference "
            f"(€{CARBON_SHADOW_EUR_PER_KG:.2f}/kg; {UBA_2024_KG_CO2_PER_KWH:.3f} kg/kWh)."
        ),
    },
    {
        "MILP part": "3️⃣ Constraints",
        "What it means": "Which rules must we respect?",
        "How it works here": (
            f"Two back-to-back, non-stop, one-hour jobs; "
            f"{WORK_START}–{WORK_END}; assumed {GRID_CAP_KW:.1f} kW site limit."
        ),
    },
])

show_table(milp_explanation, "⚙️ Our MILP explained in three simple steps")

report(
    "The optimizer selects **when** both jobs start, looks for lower estimated "
    "electricity cost and grid-related CO₂, and checks the schedule follows "
    "our assumed working rules. The historical cost chart below shows how "
    "that choice compares with starting work at 08:00."
)
.

**⚙️ Our MILP explained in three simple steps**

MILP part,What it means,How it works here
1️⃣ Decision variables,What start times can we choose?,A yes/no decision for each 15-minute start time of each job.
2️⃣ Objective function,What do we want to minimize?,Estimated electricity cost + a small CO₂ preference (€0.02/kg; 0.363 kg/kWh).
3️⃣ Constraints,Which rules must we respect?,"Two back-to-back, non-stop, one-hour jobs; 08:00–18:00; assumed 4.0 kW site limit."


> **💡 What this tells us**
>
> The optimizer selects **when** both jobs start, looks for lower estimated electricity cost and grid-related CO₂, and checks the schedule follows our assumed working rules. The historical cost chart below shows how that choice compares with starting work at 08:00.

<a id="results"></a>
### 💶 5. Does flexible scheduling save electricity costs vs an 08:00 shift start?

The chart compares **🌅 starting the same two jobs at 08:00**, **⚙️ the main MILP recommendation**, and **🔭 Hindsight**. All complete **2.40 kWh of identical illustrative work**. Lower realized modeled wholesale cost is better; the MILP makes its earlier decision using risk-weighted cost and an assumed emissions preference.

**How to read it:** Blue is the assumed **08:00 shift-start baseline**. Green is the schedule selected using information available at decision time. Grey is the **perfect-information historical benchmark**, which cannot be deployed in advance.

<details><summary>💡 Why do I compare against 08:00 instead of claiming real factory savings?</summary>

The publisher does not provide actual production orders or a factory shift-start policy. **08:00 is a transparent start-of-working-day assumption**, not an observed operational decision. I can evaluate *modeled savings versus that rule*, not realized savings paid by a company.

</details>

In [8]:
# 📊 6. Three-way historical comparison: assumed 08:00 shift start, MILP, Hindsight.
labels = ["🌅 At opening · 08:00", "⚙️ Forecast-informed MILP", "🔭 Hindsight · reference"]
fields = ["Early_cost_EUR", "MILP_cost_EUR", "Hindsight_cost_EUR"]
colors = [BLUE, GREEN, SLATE]
means = [float(results.get_column(col).mean()) for col in fields]
headline = pl.DataFrame({
    "Strategy": labels,
    "Mean cost (€ / 2 jobs)": means,
    "Total cost (€)": [float(results.get_column(col).sum()) for col in fields],
})
show_table(headline, f"💶 Same workload and {results.height} historical days")

baseline_cost = means[0]
mean_saving = float(results.get_column("MILP_saving_vs_early_EUR").mean())
relative_saving = (100 * mean_saving / baseline_cost) if baseline_cost > 0 else None
better_days = results.filter(pl.col("MILP_saving_vs_early_EUR") > 1e-7).height
same_days = results.filter(pl.col("MILP_saving_vs_early_EUR").abs() <= 1e-7).height
worse_days = results.height - better_days - same_days
mean_regret = float(results.get_column("MILP_regret_EUR").mean())

show_cards([
    ("🌅 Start at opening", f"€{means[0]:.5f}", "Average wholesale cost per two jobs"),
    ("⚙️ MILP", f"€{means[1]:.5f}", "Same work, start time selected by model"),
    ("🔭 Hindsight", f"€{means[2]:.5f}", "Cheapest historical cost, known afterward"),
    ("💶 Mean saving", f"€{mean_saving:+.5f}", "Positive = MILP cheaper than 08:00"),
    ("📉 Change vs 08:00", f"{relative_saving:+.1f}%" if relative_saving is not None else "—",
     "Based on matched days, not annual savings"),
    ("✅ Days improved", f"{better_days}/{results.height}", f"{same_days} ties · {worse_days} worse"),
])

fig = go.Figure(go.Bar(
    x=labels, y=means, marker_color=colors,
    text=[f"€{v:.5f}" for v in means], textposition="outside",
))
fig.update_layout(
    title={"text": "Did starting later reduce the modeled electricity cost?"
                   "<br><sup>Same two hypothetical jobs, actual archived PV and SMARD prices · Hindsight is a reference</sup>", "x": .04},
    template="plotly_white", height=480, showlegend=False,
    margin=dict(l=70, r=25, t=100, b=75),
    font=dict(family="Arial,sans-serif", color="#334155"),
)
fig.update_yaxes(title="Mean modeled wholesale cost (€ / two jobs)", rangemode="tozero")
fig.show()

report(
    f"Across **{results.height} matched dates**, the MILP changed the modeled wholesale cost "
    f"by **€{mean_saving:+.5f} per two-job schedule** compared with the assumed 08:00 start "
    f"({f'{relative_saving:+.1f}%' if relative_saving is not None else 'percentage unavailable'}). "
    f"It was cheaper on **{better_days} days**, tied on **{same_days}**, and cost more on **{worse_days}**. "
    "These are *historical scenario estimates*, not verified factory bills or measured savings."
)

**💶 Same workload and 11 historical days**

Strategy,Mean cost (€ / 2 jobs),Total cost (€)
🌅 At opening · 08:00,0.09333,1.02661
⚙️ Forecast-informed MILP,0.04319,0.47506
🔭 Hindsight · reference,0.02989,0.32881


> **💡 What this tells us**
>
> Across **11 matched dates**, the MILP changed the modeled wholesale cost by **€+0.05014 per two-job schedule** compared with the assumed 08:00 start (+53.7%). It was cheaper on **9 days**, tied on **0**, and cost more on **2**. These are *historical scenario estimates*, not verified factory bills or measured savings.

<a id="sustainability"></a>
### 🌍 6. What is the environmental impact of moving the jobs?

A cheaper schedule may also use more solar electricity and less grid power. To compare the same work fairly, I measure the **additional grid electricity** required by each schedule, then estimate its associated CO₂ using Germany's **2024 annual electricity-mix factor**.

| Measure | How it is calculated |
|:---|:---|
| 🔌 **Additional grid electricity** | Grid energy needed for the two jobs after the assumed PV contribution (kWh) |
| 🌱 **2024 emissions factor** | **0.363 kg CO₂ per kWh** of grid electricity — German Umweltbundesamt (UBA) |
| 📊 **Estimated grid CO₂** | **Additional grid electricity (kWh) × 0.363 kg CO₂/kWh** |

**How to read the results:** All strategies perform the same **2.40 kWh of work**. A lower grid-energy value means more of that work is covered by assumed on-site PV, so the annual-factor CO₂ estimate is also lower.

<details><summary>📚 Source, method and interpretation</summary>

- **Source:** [Umweltbundesamt — CO₂ emissions per kilowatt-hour of electricity in 2024](https://www.umweltbundesamt.de/themen/co2-emissionen-pro-kilowattstunde-strom-2024), published **9 April 2025**. The same factor is used in `03_plant_optimization_eda.ipynb`.
- **Scope:** This is a **generation-related estimate** based on Germany's *annual average*. It is not an hourly emissions forecast, a lifecycle assessment or independently measured avoided CO₂.
- **Assumptions:** PV electricity can offset the jobs' grid imports only if the systems share an electrical connection. That connection is **not verified** in the dataset.
- **Optimization setting:** The MILP also includes a research weight of **€0.02 per kg CO₂**. This is an assumed preference, **not a government carbon price or customer tariff**.

</details>


In [9]:
# 🌍 7. Sustainability — modeled grid energy and UBA-2024 estimated CO₂.
energy_means = [float(results.get_column(f"{s}_grid_kWh").mean())
                for s in ("Early", "MILP", "Hindsight")]
co2_means = [float(results.get_column(f"{s}_CO2_kg").mean())
             for s in ("Early", "MILP", "Hindsight")]
energy_summary = pl.DataFrame({
    "Strategy": labels,
    "Mean grid import (kWh / 2 jobs)": energy_means,
    "Mean estimated CO₂ (kg / 2 jobs)": co2_means,
})
show_table(energy_summary, "🌱 Additional grid electricity and annual-factor CO₂ estimates")
mean_grid_saved = float(results.get_column("Grid_reduction_vs_early_kWh").mean())
mean_co2_saved = float(results.get_column("CO2_reduction_vs_early_kg").mean())

from plotly.subplots import make_subplots
fig_green = make_subplots(rows=1, cols=2, horizontal_spacing=.14,
                          subplot_titles=("Additional grid electricity", "Estimated CO₂ · UBA 2024"))
fig_green.add_trace(go.Bar(x=labels, y=energy_means, marker_color=colors,
                           text=[f"{v:.3f} kWh" for v in energy_means], textposition="outside",
                           showlegend=False), row=1, col=1)
fig_green.add_trace(go.Bar(x=labels, y=co2_means, marker_color=colors,
                           text=[f"{v:.3f} kg" for v in co2_means], textposition="outside",
                           showlegend=False), row=1, col=2)
fig_green.update_yaxes(title="Grid imports (kWh / two jobs)", rangemode="tozero", row=1, col=1)
fig_green.update_yaxes(title="Estimated CO₂ (kg / two jobs)", rangemode="tozero", row=1, col=2)
fig_green.update_layout(template="plotly_white", height=470,
                        title={"text": "How scheduling changes grid electricity and estimated CO₂"
                               "<br><sup>Same illustrative jobs · constant UBA 2024 annual factor</sup>", "x": .04},
                        margin=dict(l=75,r=25,t=105,b=85),
                        font=dict(family="Arial,sans-serif",color="#334155"))
fig_green.show()

if mean_grid_saved > 1e-9:
    energy_message = f"**{mean_grid_saved:.3f} kWh less**"
    co2_message = f"**{mean_co2_saved:.3f} kg less**"
elif mean_grid_saved < -1e-9:
    energy_message = f"**{abs(mean_grid_saved):.3f} kWh more**"
    co2_message = f"**{abs(mean_co2_saved):.3f} kg more**"
else:
    energy_message = "**the same amount of grid electricity**"
    co2_message = "**unchanged**"
report(
    f"Across **{results.height} matched dates**, MILP required {energy_message} "
    f"per two-job schedule than starting at 08:00. The annual-factor CO₂ estimate "
    f"was {co2_message}. The jobs still consume the same 2.40 kWh; "
    "only the modeled share supplied by the grid changes. "
    "These are scenario estimates, not verified avoided emissions."
)

**🌱 Additional grid electricity and annual-factor CO₂ estimates**

Strategy,Mean grid import (kWh / 2 jobs),Mean estimated CO₂ (kg / 2 jobs)
🌅 At opening · 08:00,1.24854,0.45322
⚙️ Forecast-informed MILP,1.03903,0.37717
🔭 Hindsight · reference,0.86348,0.31344


> **💡 What this tells us**
>
> Across **11 matched dates**, MILP required **0.210 kWh less** per two-job schedule than starting at 08:00. The annual-factor CO₂ estimate was **0.076 kg less**. The jobs still consume the same 2.40 kWh; only the modeled share supplied by the grid changes. These are scenario estimates, not verified avoided emissions.

<a id="schedule"></a>
### 🕒 7. When would the machine jobs run?

The first chart compares **the three modeled electricity costs by day**. The timeline then shows when the same two consecutive jobs would run under the assumed 08:00 shift start, the MILP recommendation, and Hindsight.

**In simple words:** The line graph tells us *how much the electricity would cost* each day; the timeline shows *what time the work would happen*. Hindsight is there only to show opportunities missed by imperfect forecasts.

<details><summary>🔎 Why can the model choose a different time each day?</summary>

The wholesale price and expected solar availability change across dates. But a single 15-minute morning forecast cannot tell us the cloud conditions across the whole day, so **the decision can still be wrong**. Hindsight has future measured solar and is a lower-cost reference, never an operational forecast.

</details>

In [10]:
# 🕒 8. Daily electricity costs and continuous two-hour schedules.
chart = go.Figure()
for label, column, color in zip(
    ("🌅 Company starts · 08:00", "⚙️ Forecast MILP", "🔭 Hindsight · reference"),
    fields, colors,
):
    chart.add_trace(go.Scatter(
        x=results.get_column("date").to_list(),
        y=results.get_column(column).to_list(),
        mode="lines+markers", name=label,
        line=dict(color=color, dash="dash" if "Hindsight" in label else "solid"),
    ))
chart.update_layout(
    title="Daily modeled electricity cost — did MILP improve on opening time?",
    template="plotly_white", height=420, hovermode="x unified",
    margin=dict(l=65, r=30, t=75, b=60),
)
chart.update_yaxes(title="Wholesale cost (€ / two jobs)")
chart.update_xaxes(title="Historical date")
chart.show()

example = results.row(0, named=True)
example_day = datetime.combine(example["date"], datetime.min.time())
timeline = go.Figure()
strategies = [
    ("Company opens · 08:00", EARLY_STARTS, BLUE),
    ("Forecast MILP", tuple(example["MILP_starts"].split(" + ")), GREEN),
    ("Hindsight", tuple(example["Hindsight_starts"].split(" + ")), SLATE),
]
for label, starts, color in strategies:
    for job, clock in enumerate(starts):
        hh, mm = map(int, clock.split(":"))
        begin = example_day + timedelta(hours=hh, minutes=mm)
        finish = begin + timedelta(minutes=15 * JOB_STEPS[job])
        timeline.add_trace(go.Scatter(
            x=[begin, finish], y=[label, label], mode="lines+markers",
            line=dict(color=color, width=16),
            marker=dict(size=7, color=color),
            opacity=.75 if label == "Hindsight" else 1.0,
            showlegend=False,
            hovertemplate=f"{label} · Job {job+1}<extra></extra>",
        ))
timeline.update_layout(
    title=f"When would the two jobs run? · {example_day:%d %b %Y}",
    template="plotly_white", height=355,
    margin=dict(l=100, r=25, t=75, b=55),
    xaxis=dict(title="Start / finish", range=[example_day + timedelta(hours=7, minutes=45),
                                               example_day + timedelta(hours=18, minutes=15)],
               tickformat="%H:%M"),
    yaxis=dict(title="Strategy", categoryorder="array",
               categoryarray=["Hindsight", "Forecast MILP", "Company opens · 08:00"]),
)
timeline.show()

show_table(results.select([
    "date", "period", "Early_starts", "MILP_starts", "Hindsight_starts",
    "Early_cost_EUR", "MILP_cost_EUR", "Hindsight_cost_EUR",
    "MILP_saving_vs_early_EUR", "MILP_regret_EUR",
    "Early_grid_kWh", "MILP_grid_kWh", "Hindsight_grid_kWh",
    "Early_CO2_kg", "MILP_CO2_kg", "Hindsight_CO2_kg",
]), "📋 Daily schedules, modeled costs, grid use and sustainability", max_rows=40)

period_summary = results.group_by("period", maintain_order=True).agg([
    pl.len().alias("Days"),
    pl.col("Early_cost_EUR").mean().alias("08_mean_EUR"),
    pl.col("MILP_cost_EUR").mean().alias("MILP_mean_EUR"),
    pl.col("Hindsight_cost_EUR").mean().alias("Hindsight_mean_EUR"),
    pl.col("Early_grid_kWh").mean().alias("08_grid_kWh"),
    pl.col("MILP_grid_kWh").mean().alias("MILP_grid_kWh"),
    pl.col("Early_CO2_kg").mean().alias("08_estimated_CO2_kg"),
    pl.col("MILP_CO2_kg").mean().alias("MILP_estimated_CO2_kg"),
])
show_table(period_summary, "☀️ Unusual zero-PV readings — shown separately")
report("The schedule is evaluated on real archived solar readings and market prices, "
       "but the two jobs and their flexibility remain **illustrative assumptions**. "
       "The prolonged April zero-PV period is kept separate rather than hidden.")

**📋 Daily schedules, modeled costs, grid use and sustainability**

date,period,Early_starts,MILP_starts,Hindsight_starts,Early_cost_EUR,MILP_cost_EUR,Hindsight_cost_EUR,MILP_saving_vs_early_EUR,MILP_regret_EUR,Early_grid_kWh,MILP_grid_kWh,Hindsight_grid_kWh,Early_CO2_kg,MILP_CO2_kg,Hindsight_CO2_kg
25 Mar 2024,Outside extended zero-PV period,08:00 + 09:00,11:00 + 12:00,10:15 + 11:15,0.03499,0.00000,0.00000,0.03499,0.00000,0.41826,0.00000,0.00000,0.15183,0.00000,0.00000
26 Mar 2024,Outside extended zero-PV period,08:00 + 09:00,12:00 + 13:00,09:15 + 10:15,0.04998,0.00000,0.00000,0.04998,0.00000,0.70190,0.00000,0.00000,0.25479,0.00000,0.00000
27 Mar 2024,Outside extended zero-PV period,08:00 + 09:00,10:45 + 11:45,12:00 + 13:00,0.02570,0.06974,0.00984,-0.04404,0.05989,0.33362,1.19239,0.17036,0.12111,0.43284,0.06184
28 Mar 2024,Outside extended zero-PV period,08:00 + 09:00,13:00 + 14:00,09:15 + 10:15,0.06700,0.00000,0.00000,0.06700,0.00000,0.85382,0.00000,0.00000,0.30994,0.00000,0.00000
29 Mar 2024,Outside extended zero-PV period,08:00 + 09:00,12:00 + 13:00,10:00 + 11:00,0.01924,0.01720,0.00230,0.00204,0.01490,1.23735,1.56267,0.13601,0.44916,0.56725,0.04937
01 Apr 2024,Outside extended zero-PV period,08:00 + 09:00,13:00 + 14:00,15:00 + 16:00,0.11079,0.00001,-0.00002,0.11078,0.00002,1.85308,0.29776,1.54346,0.67267,0.10809,0.56028
02 Apr 2024,Outside extended zero-PV period,08:00 + 09:00,11:15 + 12:15,10:30 + 11:30,0.01358,0.00000,0.00000,0.01358,0.00000,0.26315,0.00000,0.00000,0.09552,0.00000,0.00000
03 Apr 2024,Outside extended zero-PV period,08:00 + 09:00,12:00 + 13:00,10:15 + 11:15,0.09126,0.09323,0.04135,-0.00196,0.05188,0.87275,1.17656,0.44840,0.31681,0.42709,0.16277
04 Apr 2024,Apr 4–8 extended zero-PV readings,08:00 + 09:00,11:00 + 12:00,14:00 + 15:00,0.17894,0.14537,0.13399,0.03358,0.01138,2.40000,2.40000,2.40000,0.87120,0.87120,0.87120
05 Apr 2024,Apr 4–8 extended zero-PV readings,08:00 + 09:00,13:00 + 14:00,13:00 + 14:00,0.17831,-0.00012,-0.00012,0.17843,0.00000,2.40000,2.40000,2.40000,0.87120,0.87120,0.87120


**☀️ Unusual zero-PV readings — shown separately**

period,Days,08_mean_EUR,MILP_mean_EUR,Hindsight_mean_EUR,08_grid_kWh,MILP_grid_kWh,08_estimated_CO2_kg,MILP_estimated_CO2_kg
Outside extended zero-PV period,8,0.05157,0.02252,0.00668,0.81674,0.52867,0.29648,0.19191
Apr 4–8 extended zero-PV readings,3,0.20470,0.09830,0.09178,2.40000,2.40000,0.87120,0.87120


> **💡 What this tells us**
>
> The schedule is evaluated on real archived solar readings and market prices, but the two jobs and their flexibility remain **illustrative assumptions**. The prolonged April zero-PV period is kept separate rather than hidden.

<a id="takeaways"></a>
### 💡 8. What did I learn, and what would a real factory still need?

**✅ What I can show:** One MILP compares flexible starts with **running the same work at the assumed 08:00 opening time**. I measure modeled wholesale electricity costs, additional grid energy and **estimated CO₂**, using real archived PV and price records.

**🛠️ Monitoring:** Notebook 02 adds time-aware quality checks and review observations to each evaluated scheduling day. They are advisory only; they do not authorize automatic stops or prove machine degradation.

**🌍 Environmental impact:** The CO₂ estimate follows **modeled additional grid imports** using the UBA 2024 annual factor. This is an energy-source comparison, **not an hourly carbon forecast** or proof of actual avoided emissions.

**🔭 What Hindsight teaches us:** It reveals the cheapest historical schedule *after* solar production is observed. The gap to MILP measures how much better timing information might help.

**⚠️ What I cannot claim:** The publisher does not supply verified factory working hours, moveable work orders, grid limits or PV connection. The two jobs and 08:00 opening time are **scenario assumptions**, and SMARD wholesale prices are **not the factory electricity bill**. Solar measurements end in April 2024; no full-year savings are established.

<details><summary>🛡️ What would make this usable in an actual factory?</summary>

- 🔌 Confirm the real site connection and spare capacity.
- 🧾 Obtain job orders, durations, deadlines and authorized start times.
- ☁️ Use full-horizon weather/PV forecasts available at each decision.
- 🔄 Re-optimize unfinished jobs when better information arrives (not implemented here).
- 🌱 Add an hourly emissions signal if carbon-aware timing itself is a business objective.

</details>

In [11]:
# 💾 9. Export the three-strategy historical study, including sustainability.
summary_file = OUT / "milp_final_opening_vs_milp_vs_hindsight.csv"
daily_file = OUT / "milp_final_opening_daily_decisions.csv"
manifest_file = OUT / "milp_final_opening_assumptions.json"
headline.write_csv(summary_file)
results.write_csv(daily_file)
with manifest_file.open("w", encoding="utf-8") as f:
    json.dump({
        "study": "Historical SPARK/SMARD what-if: assumed 08:00 shift vs MILP vs Hindsight",
        "strategies": ["assumed opening 08:00", "forecast-informed MILP", "hindsight reference"],
        "work_start": WORK_START, "work_end": WORK_END,
        "job_power_kw": list(JOB_KW), "job_duration_slots": list(JOB_STEPS),
        "job_sequence": "two_back_to_back_jobs_without_idle_gap",
        "total_illustrative_work_kwh": TOTAL_JOB_KWH,
        "early_start_assumption": list(EARLY_STARTS),
        "solar_scenario_weights": list(RISK_WEIGHTS),
        "carbon_preference_eur_per_kg_assumed": CARBON_SHADOW_EUR_PER_KG,
        "UBA_2024_annual_factor_kg_CO2_per_kwh": UBA_2024_KG_CO2_PER_KWH,
        "UBA_reference": UBA_SOURCE_URL,
        "min_pv_coverage": MIN_PV_COVERAGE,
        "matched_days": results.height,
        "skipped_dates": skipped,
        "mean_saving_vs_08_eur_per_two_jobs": mean_saving,
        "mean_grid_reduction_vs_08_kwh_per_two_jobs": mean_grid_saved,
        "mean_co2_reduction_vs_08_kg_per_two_jobs": mean_co2_saved,
        "mean_regret_vs_hindsight_eur_per_two_jobs": mean_regret,
        "hindsight_is_not_operational": True,
        "hindsight_uses_future_recorded_pv": True,
        "pv_data_ends_april_2024": True,
        "forecast_horizon_minutes": 15,
        "forecast_available_at": "08:00 (assuming the 07:45–08:00 reading is ready immediately)",
        "forecast_predicted_interval": "08:00–08:15",
        "forecast_source_file": str(FORECAST_FILE.relative_to(ROOT)),
        "forecast_prediction_column": selected_prediction,
        "notebook_02_advisory_file": (str(NOTEBOOK2_FILE.relative_to(ROOT)) if NOTEBOOK2_FILE else None),
        "notebook_02_monitoring_cutoff": "available_at <= 08:00 and interval_end <= 08:00",
        "notebook_02_lookback_hours": MONITOR_LOOKBACK_HOURS,
        "notebook_02_max_age_minutes": MONITOR_MAX_AGE_MINUTES,
        "notebook_02_warnings_are_advisory_only": True,
        "notebook_02_scores_change_milp_objective": False,
        "remaining_39_intervals_use_prior_pv_profiles": True,
        "full_workday_forecast_available": False,
        "electricity_prices_are_archived_wholesale": True,
        "published_shift_start_is_verified": False,
        "factory_work_orders_are_verified": False,
        "PV_electrical_sharing_is_verified": False,
        "actual_factory_bill_savings_are_verified": False,
    }, f, indent=2, ensure_ascii=False)
show_table(pl.DataFrame({
    "Saved file": [p.name for p in (summary_file, daily_file, manifest_file)],
    "Why it matters": ["Three-strategy cost comparison",
                       "Daily timing, monitoring context, energy, cost and CO₂",
                       "Reproducible assumptions and scientific limitations"],
}), "💾 Historical study exports · notebook reports only")
report("The historical comparison, daily modeled electricity/CO₂ results and assumptions "
       "are saved separately from your Python pipeline outputs. "
       "No synthetic factory data or weather API is used in this notebook.")

**💾 Historical study exports · notebook reports only**

Saved file,Why it matters
milp_final_opening_vs_milp_vs_hindsight.csv,Three-strategy cost comparison
milp_final_opening_daily_decisions.csv,"Daily timing, monitoring context, energy, cost and CO₂"
milp_final_opening_assumptions.json,Reproducible assumptions and scientific limitations


> **💡 What this tells us**
>
> The historical comparison, daily modeled electricity/CO₂ results and assumptions are saved separately from your Python pipeline outputs. No synthetic factory data or weather API is used in this notebook.

---

### 📚 Methods and reproducibility

**Tools:** **Polars** prepares and exports data, **SciPy/HiGHS** solves the **one main MILP**, and **Plotly** draws the existing interactive graphs. The actual data sources are processed **SPARK IPE_PV** 15-minute observations, archived **SMARD wholesale prices**, and Notebook 01's standard 15-minute PV forecast export, and Notebook 02's `anomaly_review_15min.parquet` where available. There is **no weather API, no alternative solver comparison, and no synthetic demo section**.

**Environmental methodology:** The estimate applies the [UBA 2024 annual electricity factor of 363 g CO₂/kWh](https://www.umweltbundesamt.de/themen/co2-emissionen-pro-kilowattstunde-strom-2024) (published 9 April 2025) to the incremental grid electricity associated with the modeled jobs. The approach is consistent with `03_plant_optimization_eda.ipynb` and is **not an hourly or measured emissions signal**.

**💡 One final reminder:** This is a **retrospective scheduling what-if study using real energy and price records**. The flexible workload, start-of-shift baseline, and PV connection are unverified assumptions—not actual production-dispatch evidence.

<details><summary>📖 Technical reference and Hindsight</summary>

[SciPy MILP documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html).

Hindsight optimizes after the day is over using the solar measurements that are never available to the live MILP at 08:00. It is therefore a lower-cost *historical reference*, not a deployable strategy.

</details>